# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema, accessible at the following URL:

```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Create the Dataset object
dataset = mlc.Dataset(croissant_url)

# Access high-level metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their field and column `@id`s.

*All entities are referenced by their `@id` as per the Croissant schema.*

In [ ]:
# List all record sets and show their fields and columns with @id
record_sets = list(dataset.record_sets)
print(f"This dataset contains {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    print(f"  name: {rs.get('name', '[no name]')}")
    print(f"  description: {rs.get('description', '[no description]')}")
    # Fields (logical)
    fields = rs.get('field', [])
    # If single field, wrap in list
    if isinstance(fields, dict):
        fields = [fields]
    print(f"  Fields:")
    for fld in fields:
        fid = fld['@id'] if isinstance(fld, dict) and '@id' in fld else str(fld)
        print(f"    - {fid}")
    # Columns (physical)
    columns = rs.get('column', [])
    if isinstance(columns, dict):
        columns = [columns]
    if columns:
        print(f"  Columns:")
        for col in columns:
            cid = col['@id'] if isinstance(col, dict) and '@id' in col else str(col)
            print(f"    - {cid}")
    print()

## 3. Data Extraction
Load data from each record set to a `DataFrame` for analysis.
Use the record set and field `@id`s from the previous section.

*Entities are referenced by their `@id`.*

In [ ]:
# Collect all record set @id's
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    # mlcroissant expects the record_set arg to be the @id
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            print(f"Loaded DataFrame for RecordSet {rs_id} with shape {df.shape}")
            dataframes[rs_id] = df
        else:
            print(f"No records found for RecordSet {rs_id}.")
    except Exception as ex:
        print(f"Error loading records for RecordSet {rs_id}: {ex}")

# For demonstration, pick the first non-empty record set
for rs_id, df in dataframes.items():
    print(f"\nColumns for RecordSet {rs_id}:\n{df.columns.tolist()}")
    display(df.head())
    break  # Only show the first non-empty one

## 4. Exploratory Data Analysis (EDA)
We demonstrate data processing: filtering, normalization, grouping, and removing outliers, using field and record set `@id` references.

*Replace the IDs with the relevant ones for your use case as discovered above.*

In [ ]:
# -- Replace these IDs to match your dataset. --
# Example: pick first available DataFrame and numeric field
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    # Guess numeric fields by dtype
    numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]  # select first numeric
        threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 0
        print(f"Using RecordSet @id: {record_set_id}")
        print(f"Using numeric field @id: {numeric_field_id}, threshold: {threshold}")
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold} (count={len(filtered_df)}):")
        display(filtered_df.head())

        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Try grouping by a non-numeric field
        group_candidates = [col for col in df.columns if pd.api.types.is_object_dtype(df[col])]
        if group_candidates:
            group_field_id = group_candidates[0]
            print(f"\nGrouping by field @id: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
    else:
        print("No numeric columns found in the record set.")
else:
    print("No record sets with data found.")

## 5. Visualization
Visualize the distribution of the numeric field (e.g., histogram) and, if a grouping field is available, a bar chart of means by category.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    df = dataframes[record_set_id]
    if numeric_candidates:
        # Histogram of the numeric field
        plt.figure(figsize=(7,4))
        sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.show()
        # If grouping by category is available
        if group_candidates:
            grouped_df = df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            plt.figure(figsize=(10,4))
            sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
            plt.title(f"Mean {numeric_field_id} by {group_field_id}")
            plt.ylabel(f"Mean {numeric_field_id}")
            plt.xticks(rotation=45)
            plt.show()

## 6. Conclusion
This notebook demonstrated how to load, inspect, filter, normalize, and visualize data from a FAIR² dataset using the `mlcroissant` library. Record sets, fields, and columns are referenced by their Croissant `@id` for reproducibility and transparent data lineage.

**Next steps:** Extend this workflow to domain-specific questions, reusing or augmenting the demonstrated processing and visual patterns.